# I-PI-MS Analysis — public code release

Three-dimensional photoionization model and optional laboratory QMS comparison. Figure and table labels refer to the manuscript version associated with the original notebook.

The optical CSVs, original QMS spectra and study-specific VUV calibration factor are not distributed. See `README.md` for input preparation and limitations. Run the notebook sequentially with locally prepared optical inputs and a local calibration factor. QMS-dependent sections are skipped by default; set `RUN_QMS_ANALYSIS = True` only when all four original QMS files are available.

## Input files and model settings


In [ ]:
from pathlib import Path
import os
from dataclasses import dataclass
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d
from matplotlib.colors import LogNorm
from IPython.display import display

# Relative input paths; point DATA_DIR to the directory containing the CSVs.
DATA_DIR = Path.cwd()
SAVE_FIGURES = False  # Set True to export figures as 300-dpi PNGs.

# QMS data folders are distinct to avoid filename collisions.
QMS_FILES = {
    'xy_ei': '241230/1.csv',
    'xy_pi': '241230/4.csv',
    'mix_ei': '250129/6.csv',
    'mix_pi': '250129/5.csv',
}
OPTICAL_FILES = [
    '0-xylene_PI_CS.csv', 'acetone_PI_CS.csv', 'ethanol_PI_CS.csv',
    'xylene_abs.csv', 'acetone_abs.csv', 'ethanol_abs.csv',
    'Oxyegen_VUV.csv', 'VUV.csv',
]
# The experimental comparison is opt-in; the model does not require QMS CSVs.
RUN_QMS_ANALYSIS = False
INPUT_FILES = OPTICAL_FILES + (list(QMS_FILES.values()) if RUN_QMS_ANALYSIS else [])
missing = [f for f in INPUT_FILES if not (DATA_DIR/f).is_file()]
if missing:
    raise FileNotFoundError('Missing CSVs in '+str(DATA_DIR)+':\n'+'\n'.join(missing))
print(f'All {len(INPUT_FILES)} input CSVs found in: {DATA_DIR}')

# Model geometry grid [mm].
GRID_MM = 0.3
# Reference temperatures [K] for the two experimental runs.
TEMPERATURE_K = [298.15, 353.15, 353.15]  # o-xylene 25 C; acetone/ethanol 80 C
CHAMBER_PRESSURE_PA = {
    'o_xylene': 1.0e-2,  # N2-equivalent indicated pressure, o-xylene run
    'mixture': 3.0e-3,   # N2-equivalent indicated pressure, ethanol–acetone run
}
PUMP_NOMINAL_L_S = 520.0
# Assumed elbow conductance; the actual organic-vapor pumping speed is not calibrated.
PUMP_ELBOW_CONDUCTANCE_L_S = 12.1 * 30.0**3 / 55.0
PUMP_EFFECTIVE_L_S = 1.0/(1.0/PUMP_NOMINAL_L_S + 1.0/PUMP_ELBOW_CONDUCTANCE_L_S)
GAS_MIN_RADIUS_M = 0.5e-3
PHOTON_PATH_STEP_MM = 0.1

# Two-mesh transmission model.
QMS_MESH_MODEL = 'thick_open_area'

# Local VUV calibration: the study-specific factor is not distributed.
# Set IPI_MS_VUV_SPECTRAL_SCALE in your local environment before starting Jupyter.
# Alternatively assign your own calibrated factor below in a local, untracked copy.
# The factor maps the input spectral ordinate to irradiance [uW/(cm^2 eV)]
# at the reference plane (50 cm from the lamp window; see model geometry).
_scale_text = os.environ.get('IPI_MS_VUV_SPECTRAL_SCALE', '')
try:
    VUV_SPECTRAL_SCALE = float(_scale_text)
except ValueError:
    raise ValueError('Set IPI_MS_VUV_SPECTRAL_SCALE to your own calibrated positive factor.') from None
if not np.isfinite(VUV_SPECTRAL_SCALE) or VUV_SPECTRAL_SCALE <= 0:
    raise ValueError('IPI_MS_VUV_SPECTRAL_SCALE must be finite and positive.')
del _scale_text
# Literature-input corrections retained from the original analysis.
XY_PI_OFFSET_MB = 1.5
ACETONE_ABSORPTION_FACTOR = 1.1


def present_figure(fig, filename):
    if SAVE_FIGURES:
        fig.savefig(DATA_DIR / filename, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close(fig)

## Figure 5 — 3D PI-MS model geometry


In [ ]:
def set_axes_equal(ax):
    """Make 3D plot axes have equal scale."""
    x_limits = ax.get_xlim3d()
    y_limits = ax.get_ylim3d()
    z_limits = ax.get_zlim3d()

    x_range = abs(x_limits[1] - x_limits[0])
    y_range = abs(y_limits[1] - y_limits[0])
    z_range = abs(z_limits[1] - z_limits[0])

    max_range = max(x_range, y_range, z_range) / 2.0

    mid_x = np.mean(x_limits)
    mid_y = np.mean(y_limits)
    mid_z = np.mean(z_limits)

    ax.set_xlim3d(mid_x - max_range, mid_x + max_range)
    ax.set_ylim3d(mid_y - max_range, mid_y + max_range)
    ax.set_zlim3d(mid_z - max_range, mid_z + max_range)



# Model dimensions [mm]
# Geometry
IR_rad = 11.0
xA = 56.4       # Left boundary of PI zone x [mm]
xB = xA + IR_rad  # Right boundary x [mm]
h_left  = 7.4   # Height at left edge [mm]
h_right = 8.9   # Height at right edge [mm]
x_plane = 49.4  # mm


# Linear upper-boundary slope
slope = (h_right - h_left) / (xB - xA)

# Cylinder along the y-axis
R_cyl   = 10.0 / 2.0              # Radius [mm]
x_center = (xA + xB) / 2.0        # Midpoint of cylinder diameter
z_center = 0.0                    # centered at z=0

# Model grid resolution
dx = dy = dz = GRID_MM  # [mm]

# Computational grid
x_vals = np.arange(20, xB + 10, dx)
# Coordinate span extends beyond the geometric boundary
y_vals = np.arange(-25, 15, dy)
# Grid includes the full cylinder extent
z_vals = np.arange(-20, 15, dz)

X, Y, Z = np.meshgrid(x_vals, y_vals, z_vals, indexing='ij')

# VUV beam envelope
inside_x = (X >= xA) & (X <= xB)
Ytop = h_left + slope * (X - xA)         # Maximum radius at each x
rho_perp = np.sqrt(Y**2 + Z**2)          # Radius from x-axis
inside_cone = (rho_perp <= Ytop) & (X >= x_plane)
inside_revolution = inside_x & inside_cone

# Cylinder mask
inside_cylinder = ((X - x_center)**2 + (Z - z_center)**2 <= R_cyl**2) & (Y >= -15) & (Y <= 10)

# Intersection mask
mask_intersection = inside_revolution & inside_cylinder

# Coordinates for visualization
X_cone = X[inside_cone]
Y_cone = Y[inside_cone]
Z_cone = Z[inside_cone]

X_cyl = X[inside_cylinder]
Y_cyl = Y[inside_cylinder]
Z_cyl = Z[inside_cylinder]

X_int = X[mask_intersection]
Y_int = Y[mask_intersection]
Z_int = Z[mask_intersection]

r = np.sqrt(X_int**2 + Y_int**2 + Z_int**2)

# --- Photon path intermediate coordinates ---
# Parametric distance to x-plane
s = x_plane / X_int
# Intermediate coordinates
X_mid = x_plane * np.ones_like(X_int)
Y_mid = s * Y_int
Z_mid = s * Z_int
# Path length to intermediate point
r_mid = np.sqrt(X_mid**2 + Y_mid**2 + Z_mid**2)
# Remaining path length
r_diff = r - r_mid   # Difference in path lengths

#Instrument masks for the geometry illustration
body_rad = 9
head_rad = 6.5 
QMS_rad = 12.5
QMS_hole_rad = 6
QMS_shell_rad = 12
lamp_body = (X >= 0) & (X <= x_plane-0.1) & (Y**2 + Z**2 <= body_rad**2) & (Y**2 + Z**2 >= head_rad**2)
QMS_shell = ((X - x_center)**2 + (Z - z_center)**2 <= QMS_shell_rad**2) & (Y >= -15) & (Y <= 15)

QMS_ionization_chamber = ((X - x_center)**2 + (Z - z_center)**2 <= R_cyl**2) & ((X - x_center)**2 + (Z - z_center)**2 >= (R_cyl-0.4)**2) & (Y >= -15) & (Y <= 10) # 20 V
QMS_base = ((X - x_center)**2 + Z**2 <= QMS_rad**2) & ((X - x_center)**2 + Z**2 >= R_cyl**2) &  (Y <= -15)  # 20 V
QMS_focus = ((X - x_center)**2 + Z**2 <= QMS_rad**2) & ((X - x_center)**2 + Z**2 >= QMS_hole_rad**2) & (Y >= -18.4) & (Y <= -17) # -20 V
QMS_acc = ((X - x_center)**2 + Z**2 <= QMS_rad**2) & ((X - x_center)**2 + Z**2 >= QMS_hole_rad**2) & (Y >= -20.7) & (Y <= -20.4) # -150 V

QMS_goal = ((X - x_center)**2 + Z**2 <= QMS_hole_rad**2) & (Y >= -21.) & (Y <= -20.7) # -150 V

X_QMS_ionization_chamber = X[QMS_ionization_chamber]
Y_QMS_ionization_chamber = Y[QMS_ionization_chamber]
Z_QMS_ionization_chamber = Z[QMS_ionization_chamber]

X_QMS_base = X[QMS_base]
Y_QMS_base = Y[QMS_base]
Z_QMS_base = Z[QMS_base]

X_QMS_focus = X[QMS_focus]
Y_QMS_focus = Y[QMS_focus]
Z_QMS_focus = Z[QMS_focus]

X_QMS_acc = X[QMS_acc]
Y_QMS_acc = Y[QMS_acc]
Z_QMS_acc = Z[QMS_acc]

X_QMS_goal = X[QMS_goal]
Y_QMS_goal = Y[QMS_goal]
Z_QMS_goal = Z[QMS_goal]


X_body = X[lamp_body]
Y_body = Y[lamp_body]
Z_body = Z[lamp_body]

# X_head = X[lamp_head]
# Y_head = Y[lamp_head]
# Z_head = Z[lamp_head]

X_QMS = X[QMS_base]
Y_QMS = Y[QMS_base]
Z_QMS = Z[QMS_base]

X_QMS_shell = X[QMS_shell]
Y_QMS_shell = Y[QMS_shell]
Z_QMS_shell = Z[QMS_shell]

gas_out =  ((X - x_center)**2 + (Y)**2 <= 0.2**2) & (Z >= -25) & (Z <= -15)

X_gas_out = X[gas_out]
Y_gas_out = Y[gas_out]
Z_gas_out = Z[gas_out]

# Plot
fig = plt.figure(figsize=(10, 6))
ax = fig.add_subplot(111, projection='3d')

ax.scatter(X_body, Z_body, Y_body, s=5, c='gray')
ax.scatter(X_QMS, Z_QMS, Y_QMS, s=5, c='gray')
ax.scatter(X_cone, Z_cone, Y_cone, s=1, c='violet', alpha=0.01, label='VUV beam')
ax.scatter(X_QMS_shell, Z_QMS_shell, Y_QMS_shell, s=1, c='lightgray', alpha=0.01, label='QMS ionization chamber')
ax.scatter(X_int, Z_int, Y_int, s=8, c='red', label='PI region')
ax.scatter(X_gas_out, Z_gas_out, Y_gas_out, s=8, c='gray', label='Gas inlet')

ax.scatter(X_QMS_ionization_chamber, Z_QMS_ionization_chamber, Y_QMS_ionization_chamber, s=1, c='gray', alpha=0.02)
ax.scatter(X_QMS_base, Z_QMS_base, Y_QMS_base, s=1, c='gray', alpha=0.02)
ax.scatter(X_QMS_focus, Z_QMS_focus, Y_QMS_focus, s=1, c='gray', alpha=0.02)
ax.scatter(X_QMS_acc, Z_QMS_acc, Y_QMS_acc, s=1, c='gray', alpha=0.02)
ax.scatter(X_QMS_goal, Z_QMS_goal, Y_QMS_goal, s=1, c='gray', alpha=0.02)

ax.set_xlabel('x [mm]')
ax.set_ylabel('z [mm]')
ax.set_zlabel('y [mm]')

# Use equal aspect ratio for 3D axes
max_range = np.array([X.max()-X.min(),
                      Y.max()-Y.min(),
                      Z.max()-Z.min()]).max() / 2.0

mid_x = (X.max()+X.min()) * 0.5
mid_y = (Y.max()+Y.min()) * 0.5
mid_z = (Z.max()+Z.min()) * 0.5


ax.set_xlim(20, mid_x+max_range)
ax.set_ylim(-(mid_y+max_range)*1.2, mid_y+max_range*1.2)
ax.set_zlim(-(mid_z+max_range)*1.2, mid_z+max_range*1.2)


set_axes_equal(ax)


# ax.legend()
# ax.view_init(elev=45, azim=45)
ax.view_init(elev=0, azim=0)
# ax.set_yticks([])
# ax.set_ylabel("")
ax.set_xticks([])
ax.set_xlabel("")

# ax.view_init(elev=90, azim=0)
plt.tight_layout()
present_figure(fig, "Figure5_geometry.png")


# ---- Source-vial composition and gas-flow boundary conditions ----
MOLAR_MASS_G_MOL = {'o_xylene': 106.165, 'acetone': 58.0791,
                    'ethanol': 46.0684, 'N2': 28.0134, 'O2': 31.9988}
LIQUID_DENSITY_G_ML = {'ethanol': 0.7893, 'acetone': 0.7845}
ANTOINE_NIST = {
    'acetone': (4.42448, 1312.253, -32.445),  # valid 259.16–507.60 K
    'ethanol': (5.24677, 1598.673, -46.424),  # valid 292.77–366.63 K
}

def antoine_pressure_pa(species, temperature_k):
    """Antoine vapor-pressure fit: log10(P/bar) = A - B/(T/K + C)."""
    A, B, C = ANTOINE_NIST[species]
    return 1e5 * 10**(A - B/(temperature_k + C))

# Initial vial inventory assumption: 5 mL ethanol, 5 mL acetone in 50 mL sealed vial.
n_eth_liq = 5.0 * LIQUID_DENSITY_G_ML['ethanol'] / MOLAR_MASS_G_MOL['ethanol']
n_ace_liq = 5.0 * LIQUID_DENSITY_G_ML['acetone'] / MOLAR_MASS_G_MOL['acetone']
x_eth_liq = n_eth_liq/(n_eth_liq+n_ace_liq)
x_ace_liq = n_ace_liq/(n_eth_liq+n_ace_liq)
T_mix = TEMPERATURE_K[1]
p_eth_vial = x_eth_liq * antoine_pressure_pa('ethanol', T_mix)
p_ace_vial = x_ace_liq * antoine_pressure_pa('acetone', T_mix)
p_air_vial = 101325.0 * T_mix/298.15  # Sealed initial air, fixed headspace volume
p_vial_total = p_eth_vial + p_ace_vial + p_air_vial

# o-Xylene at 25 C: 8.8 hPa saturated pressure / 1013 hPa atmospheric pressure.
xyl_fraction = 8.8/1013.0
SOURCE_FRACTIONS = {
    'o_xylene': {'o_xylene': xyl_fraction,
                 'N2': (1-xyl_fraction)*0.79, 'O2': (1-xyl_fraction)*0.21},
    'mixture': {'acetone': p_ace_vial/p_vial_total,
                'ethanol': p_eth_vial/p_vial_total,
                'N2': p_air_vial/p_vial_total*0.79,
                'O2': p_air_vial/p_vial_total*0.21},
}

kB = 1.380649e-23
S_TMP = PUMP_EFFECTIVE_L_S
# Each experimental run has a different indicated chamber pressure.
P_ch_by_condition = dict(CHAMBER_PRESSURE_PA)
if set(P_ch_by_condition) != set(SOURCE_FRACTIONS):
    raise ValueError('Each source condition must have an indicated pressure.')
if any(not np.isfinite(p) or p <= 0 for p in P_ch_by_condition.values()):
    raise ValueError('Chamber pressures must be positive and finite.')
# Inferred throughput Q = S_eff * P_ch (N2-equivalent proxy, not gas-calibrated).
Q_out_by_condition = {
    condition: (S_TMP * 1e-3) * pressure_pa
    for condition, pressure_pa in P_ch_by_condition.items()
}  # Pa m^3 /s

def mean_molecular_speed_m_s(species, temperature_k):
    m_kg = MOLAR_MASS_G_MOL[species] * 1e-3/6.02214076e23
    return np.sqrt(8*kB*temperature_k/(np.pi*m_kg))

def source_field_pa(points_mm, species, condition):
    """Neutral-gas approximation with condition-specific P_ch and Q."""
    p = np.asarray(points_mm, float)
    delta = (p - np.array([x_center, 0.0, -15.0]))*1e-3
    r = np.linalg.norm(delta, axis=-1)
    r_eff = np.maximum(r, GAS_MIN_RADIUS_M)
    cos_theta = np.divide(delta[..., 2], r, out=np.zeros_like(r), where=r>0)
    cos_theta = np.clip(cos_theta, 0.0, 1.0)
    T = TEMPERATURE_K[0] if condition == 'o_xylene' else TEMPERATURE_K[1]
    v = mean_molecular_speed_m_s(species, T)
    frac = SOURCE_FRACTIONS[condition][species]
    P_ch = P_ch_by_condition[condition]
    Q_out = Q_out_by_condition[condition]
    return frac * (P_ch + Q_out*cos_theta/(np.pi*v*r_eff**2))

def species_density_cm3(points_mm, species, condition):
    T = TEMPERATURE_K[0] if condition == 'o_xylene' else TEMPERATURE_K[1]
    return source_field_pa(points_mm, species, condition)/(kB*T)*1e-6

# ---- Finite-thickness QMS mesh transmission ----
@dataclass(frozen=True)
class CylindricalMeshConfig:
    name: str
    radius_mm: float
    thickness_mm: float
    y_min_mm: float
    y_max_mm: float
    hole_axial_mm: float
    hole_circumferential_mm: float
    wire_axial_mm: float
    wire_circumferential_mm: float

    @property
    def pitch_axial_mm(self):
        return self.hole_axial_mm + self.wire_axial_mm

    @property
    def pitch_circumferential_mm(self):
        return self.hole_circumferential_mm + self.wire_circumferential_mm

    @property
    def open_area_fraction(self):
        return (
            self.hole_axial_mm / self.pitch_axial_mm
            * self.hole_circumferential_mm / self.pitch_circumferential_mm
        )


REPELLER_MESH = CylindricalMeshConfig(
    name="QMS electron repeller",
    radius_mm=12.0,
    thickness_mm=0.15,
    y_min_mm=-15.0,
    y_max_mm=15.0,
    hole_axial_mm=0.30,
    hole_circumferential_mm=0.30,
    wire_axial_mm=0.15,
    wire_circumferential_mm=0.15,
)

ION_CHAMBER_MESH = CylindricalMeshConfig(
    name="QMS ionization chamber",
    radius_mm=R_cyl,  # 5 mm outer radius
    thickness_mm=0.30,
    y_min_mm=-15.0,
    y_max_mm=10.0,
    # 1.0 mm (axial) × 2.5 mm (circumferential) clear aperture.
    hole_axial_mm=1.0,
    hole_circumferential_mm=2.5,
    wire_axial_mm=0.30,
    wire_circumferential_mm=0.30,
)

QMS_MESH_OPEN_AREA_TRANSMISSION = (
    REPELLER_MESH.open_area_fraction
    * ION_CHAMBER_MESH.open_area_fraction
)

print("QMS optical mesh geometry")
print("-------------------------")
for mesh in [REPELLER_MESH, ION_CHAMBER_MESH]:
    print(
        f"{mesh.name}: R={mesh.radius_mm:.3f} mm, t={mesh.thickness_mm:.3f} mm, "
        f"open area={mesh.open_area_fraction:.6f}"
    )
print(f"Zero-thickness combined open fraction = {QMS_MESH_OPEN_AREA_TRANSMISSION:.6f}")
print(f"Zero-thickness equivalent loss        = ×{1/QMS_MESH_OPEN_AREA_TRANSMISSION:.3f}")


def _ray_cylinder_entry_intersection(points_mm, radius_mm,
                                     x_center_mm=x_center):
    """
    First source-to-cell intersection with
        (x-x_center)^2 + z^2 = radius^2,
    for a cylinder whose axis is parallel to y.
    """
    p = np.asarray(points_mm, float)
    x = p[:, 0]
    z = p[:, 2]

    A = x*x + z*z
    B = -2.0*x_center_mm*x
    C = x_center_mm**2 - radius_mm**2

    disc = B*B - 4*A*C
    if np.any(disc < -1e-9):
        raise RuntimeError("Some VUV rays do not intersect the requested mesh cylinder.")

    root = np.sqrt(np.clip(disc, 0.0, None))
    t1 = (-B-root)/(2*A)
    t2 = (-B+root)/(2*A)

    candidates = np.column_stack((t1, t2))
    valid = (candidates > 0.0) & (candidates <= 1.0 + 1e-12)
    t = np.where(valid, candidates, np.inf).min(axis=1)

    if np.any(~np.isfinite(t)):
        raise RuntimeError("Could not identify source-to-cell mesh intersection.")

    q = p*t[:, None]
    return q, t


def _mesh_local_basis_at_entry(entry_mm, radius_mm,
                               x_center_mm=x_center):
    """
    Local cylindrical basis at the source-facing mesh surface.

    e_in : inward surface normal
    e_y  : cylinder-axis direction
    e_s  : circumferential tangent direction
    """
    q = np.asarray(entry_mm, float)

    radial = np.column_stack((
        q[:, 0] - x_center_mm,
        np.zeros(len(q)),
        q[:, 2]
    ))
    radial /= np.linalg.norm(radial, axis=1)[:, None]

    e_out = radial
    e_in = -e_out

    e_y = np.tile(np.array([0.0, 1.0, 0.0]), (len(q), 1))

    # Circumferential tangent chosen so {e_s, e_y, e_in} is locally consistent.
    e_s = np.column_stack((
        -e_out[:, 2],
        np.zeros(len(q)),
        e_out[:, 0]
    ))

    return e_in, e_y, e_s



def _mesh_ray_geometry(points_mm, mesh: CylindricalMeshConfig):
    """
    Geometry of a straight source-to-cell ray through a cylindrical mesh.

    The outer mesh surface is intersected exactly. The finite mesh thickness is then
    treated in the local tangent-plane approximation, which is appropriate because
    t/R is small (0.15/12 and 0.30/5).

    For PI cells that lie inside the radial thickness of the mesh aperture, the ray is
    followed only as far as the PI cell; it is NOT forced to cross an inner cylinder.
    This fixes the previous failure for cells with radius > (R - thickness).
    """
    p = np.asarray(points_mm, float)
    entry, t_entry = _ray_cylinder_entry_intersection(p, mesh.radius_mm)

    ray_length_to_cell_mm = np.linalg.norm(p, axis=1)
    ray_hat = p / ray_length_to_cell_mm[:, None]

    e_in, e_y, e_s = _mesh_local_basis_at_entry(entry, mesh.radius_mm)

    d_normal = np.einsum('ij,ij->i', ray_hat, e_in)
    d_axial = np.einsum('ij,ij->i', ray_hat, e_y)
    d_circ = np.einsum('ij,ij->i', ray_hat, e_s)

    if np.any(d_normal <= 0):
        raise RuntimeError("Encountered a non-inward VUV ray at a QMS mesh.")

    # Physical ray length available after crossing the outer mesh surface.
    remaining_ray_length_mm = (1.0 - t_entry) * ray_length_to_cell_mm

    # Normal depth reached by the target PI cell after the outer surface.
    normal_depth_to_cell_mm = np.maximum(
        0.0, remaining_ray_length_mm * d_normal
    )

    # Most PI cells fully cross the mesh thickness. Cells located within an aperture
    # tunnel of the mesh layer traverse only the part between the outer surface and cell.
    traversed_normal_depth_mm = np.minimum(
        mesh.thickness_mm, normal_depth_to_cell_mm
    )

    delta_axial = traversed_normal_depth_mm * d_axial / d_normal
    delta_circ = traversed_normal_depth_mm * d_circ / d_normal

    incidence_deg = np.degrees(
        np.arctan2(np.sqrt(d_axial**2 + d_circ**2), d_normal)
    )

    return {
        'entry_mm': entry,
        't_entry': t_entry,
        'd_normal': d_normal,
        'd_axial': d_axial,
        'd_circ': d_circ,
        'incidence_deg': incidence_deg,
        'traversed_normal_depth_mm': traversed_normal_depth_mm,
        'delta_axial_mm': delta_axial,
        'delta_circ_mm': delta_circ,
    }


def finite_thickness_phase_averaged_transmission(points_mm, mesh: CylindricalMeshConfig):
    """
    Phase-averaged transmission of a finite-thickness rectangular wire mesh.

    A straight ray shifts tangentially while crossing the mesh thickness. For an unknown
    random lattice phase, the usable aperture shrinks by the absolute entry-to-exit shift:

        h_eff = max(0, h - |Δu|)
        w_eff = max(0, w - |Δv|)

    For a PI cell lying inside the mesh aperture depth, only the actually traversed depth
    is used.
    """
    geom = _mesh_ray_geometry(points_mm, mesh)
    du = geom['delta_axial_mm']
    dv = geom['delta_circ_mm']
    incidence_deg = geom['incidence_deg']

    clear_axial_eff = np.clip(
        mesh.hole_axial_mm - np.abs(du), 0.0, None
    )
    clear_circ_eff = np.clip(
        mesh.hole_circumferential_mm - np.abs(dv), 0.0, None
    )

    T = (
        clear_axial_eff / mesh.pitch_axial_mm
        * clear_circ_eff / mesh.pitch_circumferential_mm
    )

    return T, du, dv, incidence_deg



PI_POINTS_MM = np.column_stack((X_int, Y_int, Z_int))
T_REPELLER, _, _, _ = finite_thickness_phase_averaged_transmission(PI_POINTS_MM, REPELLER_MESH)
T_CHAMBER, _, _, _ = finite_thickness_phase_averaged_transmission(PI_POINTS_MM, ION_CHAMBER_MESH)
QMS_MESH_TRANSMISSION = T_REPELLER * T_CHAMBER
print(f'Two-mesh phase-averaged mean T = {QMS_MESH_TRANSMISSION.mean():.5f}')

input_rows = []
for condition, fractions in SOURCE_FRACTIONS.items():
    T_gas = TEMPERATURE_K[0 if condition=='o_xylene' else 1]
    for sp, y_i in fractions.items():
        input_rows.append([condition, sp, T_gas, y_i, mean_molecular_speed_m_s(sp, T_gas)])
display(pd.DataFrame(input_rows, columns=['Run', 'Gas', 'T [K]', 'Source mole fraction', 'Mean speed [m/s]']))
print(f'S_eff = {S_TMP:.2f} L/s; pressure and inferred throughput by run:')
display(pd.DataFrame([
    {'Run': condition, 'P_ch [Pa, N2-eq]': P_ch_by_condition[condition],
     'Q [Pa m^3/s]': Q_out_by_condition[condition]}
    for condition in P_ch_by_condition
]).style.format({'P_ch [Pa, N2-eq]': '{:.3e}', 'Q [Pa m^3/s]': '{:.4e}'}))
print(f'Mixture Raoult p_acetone={p_ace_vial:.2f} Pa, p_ethanol={p_eth_vial:.2f} Pa, sealed-air p={p_air_vial:.2f} Pa')

# Total neutral pressure for the o-xylene reference run.
mol_jet = sum(source_field_pa(PI_POINTS_MM, sp, 'o_xylene') for sp in SOURCE_FRACTIONS['o_xylene'])
n_jet_cm3 = mol_jet/(kB*TEMPERATURE_K[0])*1e-6

## Figure 8 — Photoionization cross sections and VUV spectrum


In [ ]:

files = OPTICAL_FILES
names = ['o-Xylene','Acetone','Ethanol','Benzene','Acetone','Ethanol','Oxygen']
refs = ['Zhou et al., 2009','Zhou et al., 2009','Person and Nicole, 1971',
        'Rennie et al., 1998','Nobre et al., 2008',
        'Feng and Brion, 2002','Lu et al., 2010']

energy_range = np.arange(8.0, 10.7, 0.02)

def load_data(file_path):
    data = pd.read_csv(file_path, header=None)
    return data[0].values, data[1].values
def sort_by_energy(energy, cross_section):
    sorted_indices = np.argsort(energy)
    sorted_energy = energy[sorted_indices]
    sorted_cross_section = cross_section[sorted_indices]
    return sorted_energy, sorted_cross_section
def moving_average(data, window_size):
    if window_size < 2:
        return data
    pad_width = window_size // 2
    padded_data = np.pad(data, (pad_width, pad_width), mode='edge')
    smoothed_data = np.convolve(padded_data, np.ones(window_size) / window_size, mode='valid')
    return smoothed_data
def adjust_energy_and_cross_section(energy, cross_section, energy_range):
    min_energy = min(energy)
    additional_energy = np.arange(9.0, min_energy-0.2,0.2)
    additional_cross_section = np.zeros(len(additional_energy))
    adjusted_energy = np.concatenate([additional_energy, energy])
    adjusted_cross_section = np.concatenate([additional_cross_section, cross_section])
    return adjusted_energy, adjusted_cross_section
def extract_from_adjusted(energy, cross_section, energy_range):
    interpolator = interp1d(energy, cross_section, kind='linear', fill_value='extrapolate')
    extracted_cross_section = interpolator(energy_range)
    return extracted_cross_section

energy =[]
cross_section=[]
sorted_energy = []
sorted_cross_section =[]
smooth_cross_section = []
smooth_energy = []
adjusted_energy = []
adjusted_cross_section = []
extracted_cross_section = []

###Samples
for i in range(len(files)-1):
    energy.append(load_data(DATA_DIR / files[i])[0])
    cross_section.append(load_data(DATA_DIR / files[i])[1])
    # Replace negative values in sigma_E with zero
    cross_section[i] = np.where(cross_section[i] < 0, 0, cross_section[i])
    if i == 3: # benzene surrogate for o-xylene
        energy[i], cross_section[i] = sort_by_energy(1240 /energy[i], cross_section[i])
    if i == 4: #acetone
        cross_section[i] = cross_section[i] * ACETONE_ABSORPTION_FACTOR
    if i == 6: #oxygen
        energy[i], cross_section[i] = sort_by_energy(1240 /energy[i], cross_section[i])
    sorted_energy.append(sort_by_energy(energy[i], cross_section[i])[0])
    sorted_cross_section.append(sort_by_energy(energy[i], cross_section[i])[1])
    smooth_cross_section.append(moving_average(sorted_cross_section[i], window_size=3))
    smooth_energy.append(moving_average(sorted_energy[i], window_size=3))
    adjusted_energy.append(adjust_energy_and_cross_section(smooth_energy[i], smooth_cross_section[i], energy_range)[0])
    adjusted_cross_section.append(adjust_energy_and_cross_section(smooth_energy[i], smooth_cross_section[i], energy_range)[1])
    extracted_cross_section.append(extract_from_adjusted(adjusted_energy[i], adjusted_cross_section[i], energy_range))
    
# VUV lamp spectral data
vuv_data = pd.read_csv(DATA_DIR / files[-1], header=None)
energy_vuv = 1240 /vuv_data[0].values
intensity_vuv = vuv_data[1].values
sorted_energy_vuv, sorted_intensity_vuv = sort_by_energy(energy_vuv, intensity_vuv)
smooth_vuv = moving_average(sorted_intensity_vuv, window_size=3)
smooth_energy_vuv = moving_average(sorted_energy_vuv, window_size=3)
extracted_vuv = extract_from_adjusted(smooth_energy_vuv, smooth_vuv, energy_range)

extracted_vuv = VUV_SPECTRAL_SCALE * extracted_vuv
extracted_cross_section[0] = extracted_cross_section[0] - XY_PI_OFFSET_MB
extracted_cross_section[0][extracted_cross_section[0]<=0] = 0

# Plot PI cross sections and VUV irradiance
fig, ax1 = plt.subplots(figsize=(10, 6))
for i in np.arange(0,3,1):
    ax1.plot(energy_range, extracted_cross_section[i], label='{} PI Cross Section\n ({})'.format(names[i],refs[i]), linestyle='-', linewidth=2)
ax1.set_xlabel('Energy (eV)', fontsize=18)
ax1.set_ylabel('Cross Section (Mb)', fontsize=18)
ax1.tick_params(axis='y')
ax1.set_xlim([8.5,10.7])

ax2 = ax1.twinx()
ax2.plot(energy_range, extracted_vuv, label='VUV spectral irradiance\n (local calibration)', color='purple', linestyle='--', linewidth=2)
ax2.set_ylabel('Irradiance [uW/(cm2 eV)] at 50 cm', fontsize=18, color='purple')
ax2.tick_params(axis='y', labelcolor='purple')
# ax1.set_title('Ionization Cross Sections', fontsize=21)
ax1.grid(True)
ax1.tick_params(axis='both', labelsize=14)
ax2.tick_params(axis='both', labelsize=14)
fig.tight_layout()
fig.legend(loc="upper left", bbox_to_anchor=(0.08, 0.98), fontsize=13)
present_figure(fig, "Figure8_PI_cross_sections.png")

## Figure 9b — Modeled neutral-gas number-density distribution (o-xylene)


In [ ]:
from matplotlib.colors import LogNorm
import numpy as np
import matplotlib.pyplot as plt

# Cross-section through the modeled neutral-gas plume at x = x_center.
z_axis = np.arange(-32, 33, GRID_MM)
y_axis = np.arange(-32, 33, GRID_MM)
ZZ, YY = np.meshgrid(z_axis, y_axis, indexing='xy')

r_plane = np.hypot(YY, ZZ + 15) * 1e-3
cos_plane = np.divide((ZZ + 15) * 1e-3, r_plane,
                      out=np.zeros_like(r_plane), where=r_plane > 0)
valid = (cos_plane > 0) & (r_plane > 0)

# Modeled total pressure: sum over source components, each with its own v_bar(T).
points_plane = np.column_stack((np.full(YY.size, x_center), YY.ravel(), ZZ.ravel()))
p_tot = sum(source_field_pa(points_plane, sp, 'o_xylene')
            for sp in SOURCE_FRACTIONS['o_xylene']).reshape(YY.shape)

# ---- Pressure -> number density conversion ----
# n = p / (k_B T)
# [Pa] / [J K^-1 * K] = [m^-3]
# convert to [cm^-3] by dividing by 1e6
k_B = 1.380649e-23
T_std = 273.15
n_tot_cm3 = p_tot / (k_B * T_std) / 1e6

# valid領域の最小値をfloorとして使う
n_min = np.nanmin(n_tot_cm3[valid])
n_max = np.nanmax(n_tot_cm3[valid])

# 左側(~valid)も最小値で埋めて、紫で表示させる
n_plot = np.full_like(n_tot_cm3, n_min)
n_plot[valid] = n_tot_cm3[valid]

fig, ax = plt.subplots(figsize=(10, 6))

im = ax.pcolormesh(-ZZ, YY, n_plot,
                   cmap='viridis',
                   norm=LogNorm(vmin=n_min, vmax=n_max),
                   shading='auto')

ax.set(xlabel='z [mm]', ylabel='y [mm]', xlim=(-32, 32), ylim=(-32, 32))
ax.set_xlabel('z [mm]', fontsize=16)
ax.set_ylabel('y [mm]', fontsize=16)
ax.set_aspect('equal', adjustable='box')
ax.invert_xaxis()
ax.tick_params(labelsize=14)
ax.grid(True)

cb = fig.colorbar(im, ax=ax)
cb.set_label(r'$n_{\mathrm{PI},i}$ [molecules cm$^{-3}$]', fontsize=15)

fig.tight_layout()
present_figure(fig, 'Figure9_nPI_cross_section.png')

## Figure 10a — Photoionization model geometry


In [ ]:

# ============================================================
# 1. Drawing settings
# ============================================================

VIEW_ELEV = 37
VIEW_AZIM = -110
SHOW_GRID = True

# Colors
LAMP_COLOR = '#777777'
CHAMBER_COLOR = '#929292'
BEAM_COLOR = '#a23dc6'
PI_COLOR = '#d43d5c'
REP_COLOR = '#707070'
REP_ALPHA = 0.25
CHAMBER_ALPHA = 0.25
PI_ALPHA = 0.3

# Transparency
BEAM_ALPHA = 0.23

# Visual extension of VUV beam [mm]
BEAM_EXTENSION = 9.0

# ============================================================
# 2. Geometry parameters [mm]
# ============================================================

xA = 56.4
IR_rad = 11.0
xB = xA + IR_rad

h_left = 7.4
h_right = 8.9
x_plane = 49.4

slope = (h_right - h_left) / (xB - xA)

R_cyl = 5.0
x_center = (xA + xB) / 2
z_center = 0.0

# ============================================================
# 3. Figure initialization
# ============================================================

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection='3d')

# Coordinate convention for the geometry rendering:
# Physical coordinates (x, y, z)
# Plot coordinates     (x, z, y)

def surface(X, Y, Z, color, alpha, **kwargs):

    ax.plot_surface(
        X, Z, Y,
        color=color,
        alpha=alpha,
        linewidth=0,
        antialiased=True,
        shade=False,
        **kwargs
    )

def cylinder_x(x0, x1, radius, color,
               alpha=1.0, n=100):

    x, theta = np.meshgrid(
        np.linspace(x0, x1, 35),
        np.linspace(0, 2*np.pi, n),
        indexing='ij'
    )

    surface(
        x,
        radius*np.cos(theta),
        radius*np.sin(theta),
        color,
        alpha
    )

def cylinder_y(y0, y1, radius, xc, zc,
               color, alpha=1.0, n=100):

    y, theta = np.meshgrid(
        np.linspace(y0, y1, 36),
        np.linspace(0, 2*np.pi, n),
        indexing='ij'
    )

    surface(
        xc + radius*np.cos(theta),
        y,
        zc + radius*np.sin(theta),
        color,
        alpha
    )

def disk_y(y_level, radius, xc, zc,
           color, alpha, inner_radius=0):

    rho, theta = np.meshgrid(
        np.linspace(inner_radius, radius, 22),
        np.linspace(0, 2*np.pi, 100),
        indexing='ij'
    )

    x = xc + rho*np.cos(theta)
    z = zc + rho*np.sin(theta)

    surface(
        x,
        np.full_like(x, y_level),
        z,
        color,
        alpha
    )


def disk_x(x_level, radius, color, alpha=1.0, inner_radius=0.0):

    rho, theta = np.meshgrid(
        np.linspace(inner_radius, radius, 30),
        np.linspace(0, 2*np.pi, 140),
        indexing='ij'
    )

    x = np.full_like(rho, x_level)
    y = rho*np.cos(theta)
    z = rho*np.sin(theta)

    surface(x, y, z, color, alpha)

# ============================================================
# 4. VUV lamp
# ============================================================

# Lamp body cylindrical surface
cylinder_x(
    20.0, x_plane,
    9.0,
    LAMP_COLOR,
    0.96
)

# Left end face of the VUV lamp body
disk_x(
    20.0,
    9.0,
    '#707070',
    0.96
)

# Lamp front annulus
ang, rr = np.meshgrid(
    np.linspace(0, 2*np.pi, 120),
    np.linspace(6.5, 9.0, 24)
)

surface(
    np.full_like(rr, x_plane),
    rr*np.cos(ang),
    rr*np.sin(ang),
    '#696969',
    0.95
)

# ============================================================
# 5. QMS body
# ============================================================

cylinder_y(
    -35, -15,
    12.5,
    x_center, z_center,
    '#777777',
    0.73
)

disk_y(
    -15, 12.5,
    x_center, z_center,
    '#888888',
    0.72,
    inner_radius=R_cyl
)

disk_y(
    -35, 12.5,
    x_center, z_center,
    '#777777',
    0.50,
    inner_radius=R_cyl
)

# ============================================================
# 6. QMS ionization chamber
# ============================================================

# Electron Repeller
cylinder_y(
    -15, 15,
    12.0,
    x_center, z_center,
    REP_COLOR,
    REP_ALPHA
)

cylinder_y(
    -15, 10,
    R_cyl,
    x_center, z_center,
    CHAMBER_COLOR,
    CHAMBER_ALPHA
)

disk_y(
    10,
    R_cyl,
    x_center, z_center,
    '#a0a0a0',
    0.34
)

disk_y(
    -15,
    R_cyl,
    x_center, z_center,
    '#8a8a8a',
    0.15
)

# ============================================================
# 7. VUV beam
# ============================================================

x_beam_end = xB + BEAM_EXTENSION

x = np.linspace(
    x_plane,
    x_beam_end,
    140
)

theta = np.linspace(0, 2*np.pi, 140)

XX, TH = np.meshgrid(
    x, theta,
    indexing='ij'
)

beam_R = h_left + slope*(XX - xA)

surface(
    XX,
    beam_R*np.cos(TH),
    beam_R*np.sin(TH),
    BEAM_COLOR,
    BEAM_ALPHA
)

# ============================================================
# 8. PI region
# ============================================================

x_min = max(xA, x_center - R_cyl)
x_max = min(xB, x_center + R_cyl)

xs = np.linspace(x_min, x_max, 200)
u = np.linspace(-1.0, 1.0, 100)

XX = np.broadcast_to(
    xs[:, None],
    (len(xs), len(u))
)

ZZ_limit = np.sqrt(
    np.maximum(
        0,
        R_cyl**2 - (xs - x_center)**2
    )
)

B = h_left + slope*(xs - xA)

ZZ = ZZ_limit[:, None]*u[None, :]

YY_beam = np.sqrt(
    np.maximum(
        0,
        B[:, None]**2 - ZZ**2
    )
)

surface(
    XX,
    +YY_beam,
    ZZ,
    PI_COLOR,
    0.52
)

surface(
    XX,
    -YY_beam,
    ZZ,
    PI_COLOR,
    0.52
)

Y_edge = np.sqrt(
    np.maximum(
        0,
        B**2 - ZZ_limit**2
    )
)

YY_side = Y_edge[:, None]*u[None, :]

ZZ_pos = np.broadcast_to(
    +ZZ_limit[:, None],
    XX.shape
)

ZZ_neg = np.broadcast_to(
    -ZZ_limit[:, None],
    XX.shape
)

surface(
    XX,
    YY_side,
    ZZ_pos,
    PI_COLOR,
    PI_ALPHA
)

surface(
    XX,
    YY_side,
    ZZ_neg,
    PI_COLOR,
    PI_ALPHA
)

# ============================================================
# 9. Gas inlet
# ============================================================

z_tube, theta = np.meshgrid(
    np.linspace(-25, -15, 35),
    np.linspace(0, 2*np.pi, 36),
    indexing='ij'
)

x_tube = x_center + 0.4*np.cos(theta)
y_tube = 0.4*np.sin(theta)

surface(
    x_tube,
    y_tube,
    z_tube,
    '#545454',
    1.0
)

# ============================================================
# 10. Axis arrows (no text labels)
# ============================================================

ax.quiver(
    x_plane + 1.5, 0, 0,
    25, 0, 0,
    color='#b928dc',
    linewidth=3.5,
    arrow_length_ratio=0.25
)

ax.quiver(
    x_center, -15, 0,
    0, 35, 0,
    color='#176bc5',
    linewidth=3.5,
    arrow_length_ratio=0.25
)

# ============================================================
# 11. Camera and grid
# ============================================================

ax.set_xlim(19, 82)
ax.set_ylim(-29, 23)
ax.set_zlim(-29, 20)

ax.set_box_aspect((63, 52, 49))
ax.set_proj_type('ortho')

ax.view_init(
    elev=VIEW_ELEV,
    azim=VIEW_AZIM
)

if SHOW_GRID:
    ax.set_xlabel('x [mm]')
    ax.set_ylabel('z [mm]')
    ax.set_zlabel('y [mm]')
    ax.grid(True)
else:
    ax.set_axis_off()

plt.tight_layout()

present_figure(fig, 'Figure10a_geometry.png')

## Figure 11b — Radial ion-collection efficiency (SIMION)


In [ ]:
# Data
r_1 = [44,40,58,40,32,40,44,36,28,60]
r_2 = [22,12,16,16,26,22,10,20,20,20]
r_3 = [8,6,10,4,10,12,12,6,8,6]
r_4 = [4,2,2,2,4,6,4,4,2,10]
r_5 = [2,2,6,2,0,0,0,0,0,6]
r_6 = [2,2,0,0,2,0,6,0,0,0]

a_1 = [58,50,52,54,46,46,58,54,56,46]
a_2 = [10,20,30,18,20,18,20,18,24,20]
a_3 = [8,6,12,8,10,6,8,10,4,16]
a_4 = [4,0,4,0,2,4,0,0,0,4]
a_5 = [2,6,0,2,0,0,0,2,0,2]
a_6 = [0,2,2,0,0,0,0,0,0,0]

e_1 = [56,54,46,48,54,44,48,56,58,58]
e_2 = [18,14,24,22,16,16,24,22,22,28]
e_3 = [4,6,6,6,10,18,8,10,8,8]
e_4 = [4,2,2,2,0,2,2,0,0,0]
e_5 = [2,2,4,4,0,2,0,0,0,0]
e_6 = [0,2,2,2,0,2,0,0,0,0]

groups = [r_1, r_2, r_3, r_4, r_5, r_6]
groups_a = [a_1, a_2, a_3, a_4, a_5, a_6]
groups_e = [e_1, e_2, e_3, e_4, e_5, e_6]


# Bin edges and centers in r/R
edges = np.array([0.0, 0.1, 0.3, 0.5, 0.7, 0.9, 1.0])
centers = (edges[:-1] + edges[1:]) / 2

means = np.array([np.mean(g) for g in groups])
errs  = np.array([np.std(g, ddof=1) for g in groups])  # sample standard deviation
means_a = np.array([np.mean(g) for g in groups_a])
errs_a  = np.array([np.std(g, ddof=1) for g in groups_a])  # sample standard deviation
means_e = np.array([np.mean(g) for g in groups_e])
errs_e  = np.array([np.std(g, ddof=1) for g in groups_e])  # sample standard deviation




# Plot
plt.figure()
# Shaded regions

# Separator lines (explicit)
for e in edges[1:]:
    plt.axvline(e, linewidth=1, linestyle = "--")
plt.errorbar(centers-0.02, means, yerr=errs, fmt='o', capsize=5, label = "o-Xylene")
plt.errorbar(centers, means_a, yerr=errs_a, fmt='o', capsize=5, label = "Acetone")
plt.errorbar(centers+0.02, means_e, yerr=errs_e, fmt='o', capsize=5, label = "Ethanol")
plt.xlabel(r'Normalized radius $\rho$$/R$', fontsize=16)
plt.ylabel('Transmission efficiency [%]', fontsize=16)



# plt.xlabel('Group')
plt.grid(True)
    
# plt.title('Average Transmission Rate with Error Bars', fontsize=16)
plt.tick_params(axis='both', labelsize=14)
plt.xlim(0, 1.0)
plt.legend(fontsize=14,loc = "upper right")
plt.tight_layout()
present_figure(plt.gcf(), 'Figure11b_collection_efficiency.png')

## Photoionization and ion-collection calculations


In [ ]:
# Prescribed neutral supply, VUV transport, ionization, and collection.
# Optical tables, SIMION radial bins, and two-mesh transmission are input data.
h = 6.62607015e-34
c = 299792458.0
Mb_to_cm2 = 1e-18
T_std, P_std = 273.15, 101325.0
wavelength_cm = 1240.0/energy_range[::-1] * 1e-7
vtilde = 1.0/wavelength_cm
sigma_um = 1.0/(wavelength_cm*1e4)
R_std = 1.9662731e6/(22086.66-sigma_um**2) + 2.7450825e4/(133.85688-sigma_um**2)
n_std = 1.0 + R_std*1e-6
N_std = P_std/(kB*T_std)*1e-6
F_K = 1.034 + 3.17e-12*vtilde**2
cross_section_N2 = (24*np.pi**3*vtilde**4/N_std**2
                    * ((n_std**2-1)/(n_std**2+2))**2 * F_K)

# Data are ordered in increasing energy; integrate at increasing wavelength.
wavelength_nm = 1240.0/energy_range[::-1]
wavelength_m = wavelength_nm*1e-9
I0_lambda = extracted_vuv[::-1]*1e-6/(h*c)*wavelength_m

# For each condition, integrate the Species-resolved gas density along the
# photon ray from the lamp window x=49.4 mm to each PI cell (step <= 0.1 mm).
# Small chunks avoid storing the full (n_cells, n_wavelengths) arrays.
def photon_columns_cm2(condition, chunk_size=150):
    fractions = SOURCE_FRACTIONS[condition]
    T_gas = TEMPERATURE_K[0 if condition=='o_xylene' else 1]
    columns = {sp: np.empty(len(PI_POINTS_MM)) for sp in fractions}
    for first in range(0, len(PI_POINTS_MM), chunk_size):
        last = min(first+chunk_size, len(PI_POINTS_MM))
        destination = PI_POINTS_MM[first:last]
        origin = destination * (x_plane/destination[:, 0])[:, None]
        ray_mm = destination-origin
        path_len_mm = np.linalg.norm(ray_mm,axis=1)
        steps = max(2, int(np.ceil(path_len_mm.max()/PHOTON_PATH_STEP_MM))+1)
        t = np.linspace(0, 1, steps)
        samples = origin[:,None,:]+t[None,:,None]*ray_mm[:,None,:]
        sample_flat = samples.reshape(-1,3)
        ds_cm = path_len_mm*0.1/(steps-1)
        for sp in fractions:
            n = species_density_cm3(sample_flat,sp,condition).reshape(len(destination),steps)
            columns[sp][first:last] = np.trapz(n, axis=1)*ds_cm
    return columns

# Prepare the columns once per run (acetone/ethanol share the same mixture).
columns_by_condition = {
    'o_xylene': photon_columns_cm2('o_xylene'),
    'mixture': photon_columns_cm2('mixture'),
}

# geometric attenuation: Source-reference geometry
geom = (549.4/np.linalg.norm(PI_POINTS_MM,axis=1))**2
cell_volume = (GRID_MM * 0.1)**3   # mm -> cm
r_to_QMS_rad = np.sqrt((X_int-x_center)**2+Z_int**2)/(IR_rad/2)

ion_cells_by_species = []
total_ion_rate = []
total_ion_rate_in = []
model_diagnostics = []
for g, sp in enumerate(['o_xylene', 'acetone', 'ethanol']):
    condition = 'o_xylene' if g==0 else 'mixture'
    columns = columns_by_condition[condition]
    local_n = species_density_cm3(PI_POINTS_MM, sp, condition)
    sigma_pi = extracted_cross_section[g][::-1]*Mb_to_cm2
    sigma_abs = np.zeros((len(PI_POINTS_MM), len(sigma_pi)), dtype=float)
    if condition == 'o_xylene':
        sigma_abs += columns['o_xylene'][:,None]*(extracted_cross_section[3][::-1][None,:]*Mb_to_cm2)
    else:
        sigma_abs += columns['acetone'][:,None]*(extracted_cross_section[4][::-1][None,:]*Mb_to_cm2)
        sigma_abs += columns['ethanol'][:,None]*(extracted_cross_section[5][::-1][None,:]*Mb_to_cm2)
    sigma_abs += columns['O2'][:,None]*(extracted_cross_section[6][::-1][None,:]*Mb_to_cm2)
    sigma_abs += columns['N2'][:,None]*cross_section_N2[None,:]
    # Finite-thickness two-mesh factor instead of hard-coded 0.01.
    gamma = np.empty(len(PI_POINTS_MM), dtype=float)
    # sigma_abs above is a cell x lambda optical-depth matrix, used in batches below.
    for first in range(0,len(PI_POINTS_MM),500):
        last = min(first+500,len(PI_POINTS_MM))
        phi_lambda = (I0_lambda[None,:]
                      * np.exp(-sigma_abs[first:last])
                      * geom[first:last,None]
                      * QMS_MESH_TRANSMISSION[first:last,None])
        gamma[first:last] = np.trapz(sigma_pi[None,:]*phi_lambda,
                                     wavelength_nm, axis=1)
    generated = local_n*gamma*cell_volume
    ion_cells_by_species.append(generated.copy())
    total_ion_rate.append(float(generated.sum()))
    trans_bin = (means if g==0 else means_a if g==1 else means_e)*1e-2
    idx = np.abs(r_to_QMS_rad[:,None]-centers[None,:]).argmin(axis=1)
    trans = trans_bin[idx]
    detected = generated*trans
    total_ion_rate_in.append(float(detected.sum()))
    model_diagnostics.append([sp, local_n.mean(), gamma.mean(),
                              QMS_MESH_TRANSMISSION.mean(), generated.sum(), detected.sum(),
                              float(np.max(sigma_abs))])
    del sigma_abs

print('Model inputs / intermediate diagnostics (not experimental measurements):')
display(pd.DataFrame(model_diagnostics, columns=[
    'Species','Mean n_PI [cm^-3]','Mean Gamma [s^-1]','Mean mesh T',
    'Generated [ions/s]','Collected [ions/s]','Maximum optical depth']))


## Figure 10b — Spatial ion-generation rate (o-xylene)


In [ ]:
X_Z0 = X_int[(Z_int>=-0.15)&(Z_int<=0.15)]
Y_Z0 = Y_int[(Z_int>=-0.15)&(Z_int<=0.15)]
dN_ion_dt_cell_Z0 = ion_cells_by_species[0][(Z_int>=-0.15)&(Z_int<=0.15)]

fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.grid(True)

c1 = ax1.scatter(X_Z0, Y_Z0, c=dN_ion_dt_cell_Z0,s = 50, cmap="viridis")
ax1.set_xlim([54,70])
ax1.set_ylim([-10,10])

# equal aspect ratio for x and y
ax1.set_aspect('equal', adjustable='box')
ax1.set_xlabel('x [mm]', fontsize=16)
ax1.set_ylabel('z [mm]', fontsize=16)
ax1.invert_xaxis()
ax1.tick_params(axis='both', labelsize=14)

cbar1 = fig.colorbar(c1, ax=ax1)
cbar1.set_label("Ion Generation Rate [ions/s]", size=16, labelpad=10)
cbar1.ax.tick_params(labelsize=14)

fig.tight_layout()
present_figure(fig, "Figure10b_ion_generation.png")

## Figures 12–13 — EI/PI spectra and reference-scaled EI subtraction


*Optional QMS-dependent section; skipped unless `RUN_QMS_ANALYSIS` is enabled in the settings cell.*

In [ ]:
if RUN_QMS_ANALYSIS:
    # Raw QMS mass-scan CSV format.
    CSV_COLUMNS = 18
    DATA_COLUMNS = 17  # Number of numeric fields per instrument-export row
    SMOOTH_WINDOW = 5
    DOWNSAMPLE = 5
    MZ_OFFSETS = {'xy_ei': 0.5, 'xy_pi': 0.0, 'mix_ei': -1.0, 'mix_pi': -1.0}
    REFERENCE_MZ = {'xy': 32.0, 'mix': 18.0}

    def read_qms(name):
        path = DATA_DIR / QMS_FILES[name]
        df = pd.read_csv(path, header=None, encoding='cp932',
                         names=[f'col_{j}' for j in range(CSV_COLUMNS)],
                         skip_blank_lines=False)
        mz_start, mz_end = float(df.iloc[5, 0]), float(df.iloc[5, 1])
        scans, current = [], []
        for row_no in range(8, len(df)):
            row = df.iloc[row_no]
            if row.count() == 1:
                if current:
                    scans.append(np.asarray(current, dtype=float))
                    current = []
                continue
            if row.count() == 0:
                continue
            values = pd.to_numeric(row.iloc[:DATA_COLUMNS], errors='coerce').to_numpy(dtype=float)
            finite = np.flatnonzero(np.isfinite(values))
            if len(finite):
                values = values[:finite[-1]+1]
                if np.any(~np.isfinite(values)):
                    raise ValueError(f'Invalid value in {path.name}: CSV row {row_no+1}')
                current.extend(values)
        if current:
            scans.append(np.asarray(current, dtype=float))
        if not scans or len(set(map(len, scans))) != 1:
            raise ValueError(f'Invalid scan lengths in {path.name}: {[len(x) for x in scans]}')
        current_a = np.mean(scans, axis=0)
        mz = np.linspace(mz_start, mz_end, current_a.size)
        smooth = np.convolve(current_a, np.ones(SMOOTH_WINDOW)/SMOOTH_WINDOW, mode='valid')
        mz = mz[SMOOTH_WINDOW//2:SMOOTH_WINDOW//2+len(smooth)]
        return (mz[::DOWNSAMPLE][1:] + MZ_OFFSETS[name],
                smooth[::DOWNSAMPLE][1:])

    def scaled_residual(pi, ei, reference_mz):
        mz_pi, current_pi = pi
        mz_ei, current_ei = ei
        window = 1.0
        mp = np.abs(mz_pi-reference_mz) <= window
        me = np.abs(mz_ei-reference_mz) <= window
        if not np.any(mp) or not np.any(me):
            raise ValueError(f'No samples near reference m/z {reference_mz}')
        ref_pi, ref_ei = np.max(current_pi[mp]), np.max(current_ei[me])
        if ref_ei <= 0 or ref_pi <= 0:
            raise ValueError(f'Invalid reference intensity near m/z {reference_mz}')
        scale = ref_pi/ref_ei
        ei_aligned = np.interp(mz_pi, mz_ei, current_ei, left=np.nan, right=np.nan)
        residual = current_pi - scale*ei_aligned
        print(f'PI - scaled EI at m/z {reference_mz:g}: scale = {scale:.5g}')
        return mz_pi, residual

    xy_ei = read_qms('xy_ei')
    xy_pi = read_qms('xy_pi')
    mix_ei = read_qms('mix_ei')
    mix_pi = read_qms('mix_pi')
    xy_residual = scaled_residual(xy_pi, xy_ei, REFERENCE_MZ['xy'])
    mix_residual = scaled_residual(mix_pi, mix_ei, REFERENCE_MZ['mix'])

    # Figure 12 — raw EI and PI spectra, displayed with suitable current units.
    fig, axes = plt.subplots(2, 2, figsize=(16, 10), constrained_layout=True)
    raw_panels = [
        (xy_ei, axes[0, 0], 1e6, '(a) o-Xylene EI (Raw data)', 'Intensity [μA]', 110),
        (xy_pi, axes[0, 1], 1e12, '(b) o-Xylene PI (Raw data)', 'Intensity [pA]', 110),
        (mix_ei, axes[1, 0], 1e9, '(c) Acetone + Ethanol EI (Raw data)', 'Intensity [nA]', 70),
        (mix_pi, axes[1, 1], 1e12, '(d) Acetone + Ethanol PI (Raw data)', 'Intensity [pA]', 70),
    ]
    for (mz, intensity), ax, scale, title, ylabel, maximum in raw_panels:
        ax.plot(mz, intensity*scale, color='red', lw=1.8)
        ax.set(title=title, xlabel='m/z', ylabel=ylabel, xlim=(0, maximum))
        ax.title.set_fontsize(16)
        ax.title.set_fontweight('bold')
        ax.set_xticks(np.arange(0, maximum+1, 10))
        ax.tick_params(labelsize=13)
        ax.grid(alpha=0.3)
    present_figure(fig, 'Figure12_raw_spectra.png')

    # Figure 13 — residuals, not clipped at zero.
    fig, axes = plt.subplots(1, 2, figsize=(16, 5.0), constrained_layout=True)
    for (mz, intensity), ax, title, maximum in [
        (xy_residual, axes[0], '(a) o-Xylene PI (EI subtracted)', 110),
        (mix_residual, axes[1], '(b) Ethanol + Acetone PI (EI subtracted)', 70),
    ]:
        ax.plot(mz, intensity*1e12, color='red', lw=1.8)
        ax.axhline(0, color='0.45', lw=0.7, ls='--')
        ax.set(title=title, xlabel='m/z', ylabel='Intensity [pA]', xlim=(0, maximum))
        ax.title.set_fontsize(16)
        ax.title.set_fontweight('bold')
        ax.set_xticks(np.arange(0, maximum+1, 10))
        ax.tick_params(labelsize=13)
        ax.grid(alpha=0.3)
    present_figure(fig, 'Figure13_PI_minus_EI.png')

    # Run-specific count-equivalent rates (using the documented SEM-gain estimates).
    # The extraction window is re-centered on the local channel maximum.
    e = 1.602176634e-19
    G_mix = 3.0e3    # PI; mixture, 2.5 kV
    G_xy = 3.0e3      # PI; o-xylene, 2.5 kV (later in measurement sequence)
    G_EI_XY = 2.0e4  # EI; o-xylene, 2.5 kV (earlier in measurement sequence)
    G_EI_MIX = 8.0e2 # EI; mixture, 1.5 kV
    MZ_HALF_WIDTH = 0.45  # u; actual windows are centered on detected maxima
    assert 0.0 < MZ_HALF_WIDTH < 0.5



    def channel_mean_current(spectrum, target_mz, half_width=MZ_HALF_WIDTH):
        """Mean ion current [A] over a peak-centered 2*half_width window.

        The center is the maximum within nominal m/z +/- half_width.
        No negative residual values are clipped.
        """
        mz, current_a = (
            np.asarray(x, dtype=float) for x in spectrum
        )
        good = np.isfinite(mz) & np.isfinite(current_a)
        mz, current_a = mz[good], current_a[good]
        if len(mz) < 2:
            raise ValueError("Insufficient finite mass-scan samples")
        order = np.argsort(mz)
        mz, current_a = mz[order], current_a[order]
        if np.any(np.diff(mz) <= 0):
            raise ValueError("Mass coordinates must be unique")
        # Step 1: Search for the maximum around the nominal m/z
        search_mask = np.abs(mz - target_mz) <= half_width
        if not np.any(search_mask):
            raise ValueError(
                f"No samples near m/z {target_mz}"
            )
        search_indices = np.flatnonzero(search_mask)
        peak_idx = search_indices[
            np.argmax(current_a[search_mask])
        ]
        peak_center = mz[peak_idx]
        # Step 2: Recenter the integration window
        lo = peak_center - half_width
        hi = peak_center + half_width
        if lo < mz[0] or hi > mz[-1]:
            raise ValueError(
                f"Integration window outside spectrum: m/z {target_mz}"
            )
        # Step 3: Mean current with interpolated boundaries
        local_mz = np.r_[
            lo,
            mz[(mz > lo) & (mz < hi)],
            hi
        ]
        local_I = np.interp(local_mz, mz, current_a)
        integrator = (
            np.trapezoid if hasattr(np, "trapezoid")
            else np.trapz
        )
        mean_current = integrator(local_I, local_mz) / (hi - lo)
        return float(mean_current)



    def channel_rate(spectrum, target_mz, sem_gain, half_width=MZ_HALF_WIDTH):
        """SEM-gain-corrected count-equivalent rate [s^-1] for one channel."""
        if not np.isfinite(sem_gain) or sem_gain <= 0:
            raise ValueError('SEM gain must be finite and positive')
        return channel_mean_current(spectrum, target_mz, half_width)/(e*sem_gain)


    # Parent molecular-ion channels for quantitative PI analysis.
    # The observed ethanol m/z 45 residual is not assigned to direct PI.
    # Keep model species order: o-xylene, acetone, ethanol.
    PI_PARENT_MZ = {'o-Xylene': 106, 'Ethanol': 46, 'Acetone': 58}
    R_PI_MOLECULAR = {
        'o-Xylene': channel_rate(xy_residual, 106, G_xy),
        'Ethanol': channel_rate(mix_residual, 46, G_mix),
        'Acetone': channel_rate(mix_residual, 58, G_mix),
    }
    N_ion_QMS = [R_PI_MOLECULAR[name] for name in ('o-Xylene', 'Acetone', 'Ethanol')]
else:
    print("Skipped: optional QMS-dependent analysis (RUN_QMS_ANALYSIS=False).")


## Table 2 — Experimental and modeled PI detection rates


*Optional QMS-dependent section; skipped unless `RUN_QMS_ANALYSIS` is enabled in the settings cell.*

In [ ]:
if RUN_QMS_ANALYSIS:
    # Table 2 — experimental vs. modeled parent-ion PI count-equivalent rates.
    species_order = ['o-Xylene', 'Ethanol', 'Acetone']
    model_by_species = dict(zip(('o-Xylene', 'Acetone', 'Ethanol'), total_ion_rate_in))
    comparison = pd.DataFrame({
        'Species': species_order,
        'Experiment [counts/s]': [R_PI_MOLECULAR[sp] for sp in species_order],
        'Model [counts/s]': [model_by_species[sp] for sp in species_order],
    })
    comparison['Exp./Model'] = (
        comparison['Experiment [counts/s]'] / comparison['Model [counts/s]']
    )

    # Manuscript layout, display rounded to two significant figures only.
    table2 = comparison.set_index('Species').T[species_order]
    display(table2.style.format(
        lambda v: f'{v:.2g}' if np.isfinite(v) else '—'
    ).set_caption('Table 2. Parent-ion PI response: experiment vs. model'))
else:
    print("Skipped: optional QMS-dependent analysis (RUN_QMS_ANALYSIS=False).")


## Table 3 — Apparent EI/PI density sensitivity


*Optional QMS-dependent section; skipped unless `RUN_QMS_ANALYSIS` is enabled in the settings cell.*

In [ ]:
if RUN_QMS_ANALYSIS:
    # Table 3 — same PI extraction as Table 2; selected EI channels.

    EI_MZ = {
        'o-Xylene': [51, 65, 77, 91, 106],
        'Ethanol': [27, 29, 31, 45, 46],
        'Acetone': [43, 58],
    }
    EI_SPECTRUM = {
        'o-Xylene': xy_ei,
        'Ethanol': mix_ei,
        'Acetone': mix_ei,
    }
    EI_GAIN = {
        'o-Xylene': G_EI_XY,
        'Ethanol': G_EI_MIX,
        'Acetone': G_EI_MIX,
    }
    SPECIES_META = {
        'o-Xylene': ('o_xylene', 'o_xylene', TEMPERATURE_K[0]),
        'Ethanol': ('mixture', 'ethanol', TEMPERATURE_K[2]),
        'Acetone': ('mixture', 'acetone', TEMPERATURE_K[1]),
    }

    channels_rows, densities_rows, summary_rows = [], [], []
    for sp in species_order:
        condition, species_key, temp_k = SPECIES_META[sp]
        Ci = SOURCE_FRACTIONS[condition][species_key]
        P_ch = P_ch_by_condition[condition]
        Pi = Ci * P_ch
        n_cm3 = Pi/(kB*temp_k)*1e-6
        densities_rows.append({
            'Species': sp, 'Run': condition, 'C_i': Ci, 'P_ch [Pa]': P_ch,
            'T [K]': temp_k, 'P_i [Pa]': Pi, 'n_i [molecules/cm³]': n_cm3,
        })
        EI_total = 0.0
        for mz in EI_MZ[sp]:
            count_rate = channel_rate(EI_SPECTRUM[sp], mz, EI_GAIN[sp])
            EI_total += count_rate
            channels_rows.append({
                'Species': sp, 'Mode': 'EI', 'm/z': mz,
                'Rate [counts/s]': count_rate,
            })
        PI_total = R_PI_MOLECULAR[sp]  # Table 2's exact same extraction
        channels_rows.append({
            'Species': sp, 'Mode': 'PI (parent only)', 'm/z': PI_PARENT_MZ[sp],
            'Rate [counts/s]': PI_total,
        })
        K_EI = EI_total/n_cm3
        K_PI = PI_total/n_cm3
        summary_rows.append({
            'Species': sp, 'R_EI [counts/s]': EI_total,
            'R_PI [counts/s]': PI_total, 'K_EI': K_EI, 'K_PI': K_PI,
            'PI/EI': K_PI/K_EI if K_EI > 0 else np.nan,
        })

    density_df = pd.DataFrame(densities_rows)
    channels_df = pd.DataFrame(channels_rows)
    sensitivity_df = pd.DataFrame(summary_rows).set_index('Species')


    # Manuscript Table 3: no INMS values (those are discussed separately).
    table3 = pd.DataFrame({
        sp: {
            'EI rate [counts/s]': sensitivity_df.loc[sp, 'R_EI [counts/s]'],
            'EI K_det [cps/(molecule/cm³)]': sensitivity_df.loc[sp, 'K_EI'],
            'PI K_det [cps/(molecule/cm³)]': sensitivity_df.loc[sp, 'K_PI'],
            'PI/EI ratio [-]': sensitivity_df.loc[sp, 'PI/EI'],
        }
        for sp in species_order
    })
    display(table3.style.format(
        lambda v: f'{v:.2g}' if np.isfinite(v) else '—'
    ).set_caption('Table 3. Apparent selected-channel EI/PI sensitivity'))

    # Consistency of the two reported tables.
    for sp in species_order:
        exp_t2 = comparison.set_index('Species').loc[sp, 'Experiment [counts/s]']
        exp_t3 = sensitivity_df.loc[sp, 'R_PI [counts/s]']
        assert np.isclose(exp_t2, exp_t3, rtol=1e-12), sp
    assert PI_PARENT_MZ['Ethanol'] == 46
else:
    print("Skipped: optional QMS-dependent analysis (RUN_QMS_ANALYSIS=False).")


## Additional calculation — Hypothetical INMS PI-sensitivity comparison

*Supplementary scenario; not manuscript Table 4.*


*Optional QMS-dependent section; skipped unless `RUN_QMS_ANALYSIS` is enabled in the settings cell.*

In [ ]:
if RUN_QMS_ANALYSIS:
    # Table 4 — illustrative INMS PI-equivalent scenario (ethanol reference).
    # Not an instrument prediction or a calibrated cross-species sensitivity.
    K_INMS_EI = 6.0e-4  # N2, m/z 28; Waite et al. (2004) reference
    INMS_INTEGRATION_S = 0.0311
    ethanol_ratio = sensitivity_df.loc['Ethanol', 'PI/EI']
    K_INMS_PI_HYP = K_INMS_EI * ethanol_ratio

    # The one-count criterion means an expectation of one count, not guaranteed detection.
    table4 = pd.DataFrame({
        'INMS EI (N2 reference)': {
            'K_det [cps/(molecule/cm³)]': K_INMS_EI,
            'Integration time [ms]': INMS_INTEGRATION_S * 1e3,
            'One-count density [molecules/cm³]': 1 / (K_INMS_EI * INMS_INTEGRATION_S),
        },
        'Hypothetical INMS PI (ethanol ratio)': {
            'K_det [cps/(molecule/cm³)]': K_INMS_PI_HYP,
            'Integration time [ms]': INMS_INTEGRATION_S * 1e3,
            'One-count density [molecules/cm³]': 1 / (K_INMS_PI_HYP * INMS_INTEGRATION_S),
        },
    })
    display(table4.style.format(
        lambda v: f'{v:.2g}' if np.isfinite(v) else '—'
    ).set_caption('Table 4. INMS sensitivity reference and illustrative PI equivalence'))
else:
    print("Skipped: optional QMS-dependent analysis (RUN_QMS_ANALYSIS=False).")


## Additional calculation — Species densities and mass-channel contributions


*Optional QMS-dependent section; skipped unless `RUN_QMS_ANALYSIS` is enabled in the settings cell.*

In [ ]:
if RUN_QMS_ANALYSIS:
    # Supporting evidence for Table 3 (full precision retained in the DataFrames).
    display(density_df.style.format({
        'C_i': '{:.5g}', 'P_ch [Pa]': '{:.3e}',
        'T [K]': '{:.2f}', 'P_i [Pa]': '{:.3e}',
        'n_i [molecules/cm³]': '{:.3e}',
    }).set_caption('Inferred chamber-based number densities (not measured ion-source densities)'))
    display(channels_df.style.format({
        'Rate [counts/s]': '{:.3e}',
    }).set_caption('Selected mass-channel count-equivalent rates'))
    print('Rate agreement between Tables 2 and 3: verified for all three molecules.')
else:
    print("Skipped: optional QMS-dependent analysis (RUN_QMS_ANALYSIS=False).")
